# Notebook 01 — Environment Setup

**Week 2 | No toolkit required**

By the end of this notebook you will have a working LLM development environment on either Google Colab or your local machine, and will have made your first successful API call.

## Path A — Google Colab (no GPU required)

Colab provides a free T4 GPU (15 GB VRAM). It runs in your browser — no installation needed.

1. Go to https://colab.research.google.com
2. File → New notebook
3. Runtime → Change runtime type → T4 GPU → Save

In [ ]:
# Verify GPU in Colab
import torch
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}, "
          f"{torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("No GPU — check Runtime > Change runtime type")

### Installing Ollama in Colab

Colab doesn't have Ollama pre-installed, but you can install and run it in a background process:

In [ ]:
# Install and start Ollama in Colab
import subprocess, time, requests

subprocess.Popen("curl -fsSL https://ollama.com/install.sh | sh",
                 shell=True).wait()
subprocess.Popen("ollama serve", shell=True)  # background daemon
time.sleep(3)

# Pull a model that fits the T4
subprocess.run("ollama pull qwen3.5:9b", shell=True)
print("Ollama ready.")

### API Keys in Colab

> ⚠️ **Never hardcode API keys in notebook cells you might share.**

Use Colab Secrets (🔑 icon in the left sidebar):
1. Add secret `ANTHROPIC_API_KEY` with your key
2. Toggle 'Notebook access' on
3. Access it in code:

In [ ]:
# Colab secret access
try:
    from google.colab import userdata
    anthropic_key = userdata.get('ANTHROPIC_API_KEY')
    print("Key loaded from Colab secrets.")
except ImportError:
    # Not in Colab — use environment variable instead
    import os
    anthropic_key = os.getenv('ANTHROPIC_API_KEY')
    print("Key loaded from environment.")

### Colab Limitations

| Limit | Value | Workaround |
|---|---|---|
| Session timeout | 12 hours max, 90 min idle | Save work to Drive |
| GPU availability | Shared, may queue | Try off-peak hours or Colab Pro |
| Storage | Lost on disconnect | Mount Google Drive |

**Mounting Drive for persistence:**

In [ ]:
# Mount Google Drive (Colab only)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    import os
    os.makedirs('/content/drive/MyDrive/llm_course', exist_ok=True)
    print("Drive mounted at /content/drive")
except ImportError:
    print("Not in Colab — skipping Drive mount.")

## Path B — Local Setup (GPU recommended)

### Step 1: Create a virtual environment

```bash
python3 -m venv ~/ai-env
source ~/ai-env/bin/activate
```

### Step 2: Install Ollama

```bash
# Linux / macOS
curl -fsSL https://ollama.com/install.sh | sh

# Windows: download installer from https://ollama.com/download
```

### Step 3: Pull a model

Choose based on your VRAM:

```bash
ollama pull qwen3.5:9b        # 8 GB VRAM
ollama pull qwen3.6:35b-a3b   # 16 GB VRAM (MoE, efficient)
ollama pull qwen3.6:27b       # 24 GB VRAM (best coding quality)
```

### Step 4: Verify Ollama is running

```bash
ollama list          # shows downloaded models
ollama run qwen3.5:9b "Hello"  # quick smoke test
```

### Step 5: Install the ai_tools packages

```bash
cd ~/ai_tools
make install
pip install jupyter
```

All course notebooks use packages from this monorepo. The `make install` step installs them as editable packages into your venv so imports work without any path manipulation.

## Verifying Your Setup

In [ ]:
# Run from either Colab or local
import requests

MODEL = "qwen3.5:9b"  # adjust to what you pulled

try:
    response = requests.post(
        "http://localhost:11434/v1/chat/completions",
        json={
            "model": MODEL,
            "messages": [{"role": "user", "content": "Say 'setup complete' and nothing else."}],
            "temperature": 0.0
        },
        timeout=60
    )
    response.raise_for_status()
    print("Response:", response.json()["choices"][0]["message"]["content"])
    print("\n✓ Ollama is working.")
except Exception as e:
    print(f"✗ Ollama not reachable: {e}")
    print("Check that 'ollama serve' is running.")

In [ ]:
# Check ai_tools packages are installed
packages = [
    "llm_harness_core", "llm_engines", "engram_lite",
    "llm_inspector", "rag_lib", "agent_lib"
]
for pkg in packages:
    try:
        __import__(pkg)
        print(f"  ✓ {pkg}")
    except ImportError:
        print(f"  ✗ {pkg} — run 'make install' from ~/ai_tools")

## Security Checklist

Before writing any LLM application code:

- [ ] API keys are in environment variables or Colab Secrets — never in code
- [ ] `.env` files are in `.gitignore`
- [ ] LLM-generated code that will be executed runs in a container (covered in notebook 08)
- [ ] You understand what data leaves your machine when using cloud APIs

---
**Next:** [Notebook 02 — Engine Basics](02_engine_basics.ipynb)